# Step 17.2 — Flatten B×T Supervision

## 把每个位置的预测与目标，按相同顺序排成监督表

17.1 已经把 MiniGPT 的 raw logits `(B,T,V)` 与 shifted integer targets `Y:(B,T)` 接成逐位置交叉熵。本节继续解释：

- 为什么把两个位置轴 B、T 合成一个长度为 `N=B*T` 的轴；
- 怎样保持每行 logits 与正确 target 的对应关系；
- 为什么同一个 batch 包含 N 个 token-level targets，以及它们怎样组成一个等权平均损失。

核心接口是 `(B,T,V) → (B*T,V)`、`(B,T) → (B*T,)`。这里只组织监督与损失，不实现训练步骤，完成后停在17.2，不进入17.3。

材料审核方已独立核验17.1实际母版、文档、固定MLX版本接口和完整教学回复，核心静态PASS，实际修正0/2。这是材料验收，不是学员掌握或运行反馈。17.2本次初版实际修正为 **0/2**。

本母版所有代码单元均未执行、无输出；本轮只准备与发布材料，没有读取或下载语料，也没有执行Notebook、测试、训练、实验、采样或用户work副本。

## 0. 新 Work Copy：本节自带哪些依赖？

继续使用项目已有的 Python 3.12 / `.venv` kernel 和固定依赖 `mlx==0.32.2`。本节只导入：

- `mlx.core as mx`：数组、reshape与算术平均；
- `mlx.nn as nn`：沿用17.1的交叉熵接口。

新副本自带imports、两行toy chunks、手工raw logits和全部后续变量定义。以后使用时，从下方imports开始按顺序使用本节cells即可；不依赖72副本的kernel变量，不需要 `%run` 旧课，不调用sampler，不读取语料缓存，也不需要构造MiniGPT或运行forward。

手工logits只模拟真实输出的shape与接口，不是实际模型输出。相关新API会在使用前解释。

In [ ]:
import mlx.core as mx
import mlx.nn as nn

# Imports for a future work copy. No lesson cell has been executed here.


## 1. Token-Level Supervision：一个位置对应一个正确类别

**Supervision（监督）** 在这里是给某个预测位置提供正确的next-token ID。**Token-level target（token级目标）** 指每一个位置都有自己的正确类别，而不是整条序列只有一个标签。

沿用17.1的接口：

| 对象 / 符号 | 含义 |
|---|---|
| B | batch中的序列行数 |
| T | 每行输入位置数 |
| V | vocabulary size，词表类别数 |
| `logits[b,t,:]` | 位置(b,t)的V个raw scores，shape `(V,)` |
| `Y[b,t]` | 同一位置正确的next-token整数ID |
| \(\ell_{b,t}\) | 这一位置的交叉熵，即17.1的负对数似然 |

本节假定B、T为正，每个位置都有效并等权参与监督，target IDs在 `0…V-1` 内。定义：

$$
N=B\times T.
$$

N数的是**监督位置的数量**。即使多个位置的正确ID相同，它们仍是不同位置的目标；这个计数也没有断言这些位置在统计意义上互相独立。

Y已经由shift关系得到，展平时不再右移。模型产生的logits仍各自对应原有序列上下文。

## 2. 小例：两行序列，每行两个预测位置

取B=2、T=2、V=3。为每行手工给出T+1个token IDs，再按已经介绍过的切片关系取X、Y：

```text
toy chunks = [[0, 2, 1],
              [1, 0, 2]]

X = [[0, 2],      Y = [[2, 1],
     [1, 0]]           [0, 2]]
```

每行都在自己的chunk内右移一次；第二行的目标不是从第一行继续切出来的。共有2×2=4个监督位置。

下面给每个位置手工指定一组logits。记 \(\ln[a,b,c]\) 为 \([\ln a,\ln b,\ln c]\)，即逐元素取自然对数：

| 原位置(b,t) | 正确Y | 手工logits |
|---|---:|---|
| (0,0) | 2 | \(\ln[1,2,3]\) |
| (0,1) | 1 | \(\ln[3,2,1]\) |
| (1,0) | 0 | \(\ln[1,1,2]\) |
| (1,1) | 2 | \(\ln[3,2,1]\) |

第一行直接承接17.1的两个位置；增加第二行，是为了看清B轴与T轴怎样共同组织。

### 代码前：数组定义、shape与变量

`mx.array(..., dtype=mx.int32)` 保存整数token IDs；`mx.float32` 用于浮点logits。`mx.log` 对正数逐元素取自然对数，只是为了构造可手算的分数，不是对token IDs计算logits。

`B, T, V = toy_logits.shape` 是Python的按顺序解包：把三条轴的长度分别赋给B、T、V。`N = B * T` 得到监督位置总数。这个示例不调用模型、随机数或sampler。

In [ ]:
# USER-RUN TOY SETUP — prepared only, NOT executed in the master.

toy_chunks = mx.array(
    [[0, 2, 1], [1, 0, 2]],
    dtype=mx.int32,
)  # (B, T+1) = (2, 3)
X = toy_chunks[:, :-1]  # (2, 2)
Y = toy_chunks[:, 1:]   # (2, 2), already shifted

toy_logits = mx.log(mx.array(
    [
        [[1.0, 2.0, 3.0], [3.0, 2.0, 1.0]],
        [[1.0, 1.0, 2.0], [3.0, 2.0, 1.0]],
    ],
    dtype=mx.float32,
))  # (B, T, V) = (2, 2, 3)

B, T, V = toy_logits.shape
N = B * T

print('X/Y shapes:', X.shape, Y.shape)
print('logits shape:', toy_logits.shape)
print('number of supervised positions:', N)

# Explicit toy inputs only; no corpus access or model forward is needed.


## 3. Flatten：合并B、T，保留完整的类别轴V

**Flatten（展平）** 在本节特指：把B、T这两个位置轴合成一个轴，按顺序列出每个位置的一整组V个分数。

可以把原张量理解为B张表，每张表有T行，每行有V个候选分数。现在把这些表的行依次排到同一张监督表上，得到N行、每行V列。

定义展平后的行号r，从0开始。原数组按B、T的顺序组织位置；同一行中的t先变化，经过T个位置后再进入下一个b：

$$
r=bT+t,\qquad 0\le r<N.
$$

正确的对应关系是：

$$
\text{flat\_logits}[r,v]=\text{logits}[b,t,v],
\qquad
\text{flat\_targets}[r]=Y[b,t].
$$

| 展平行号r | 原位置(b,t) | 保留的整组分数 | 对应target |
|---:|---|---|---:|
| 0 | (0,0) | \(\ln[1,2,3]\) | 2 |
| 1 | (0,1) | \(\ln[3,2,1]\) | 1 |
| 2 | (1,0) | \(\ln[1,1,2]\) | 0 |
| 3 | (1,1) | \(\ln[3,2,1]\) | 2 |

因此，展平后的targets按同样顺序为 `[2,1,0,2]`。最后一轴V一直保留：每行仍然要在V个候选类别中识别对应target。

这里重新组织的是已经得到的预测结果与监督目标；X继续保持原来的 `(B,T)` 模型输入接口。展平不会让第二行序列获得第一行的上下文，也不改变原先的因果关系。

### 代码前：`mx.reshape` 与一维shape的逗号

**Reshape（重塑形状）** 按数组的逻辑元素顺序改变维度组织，元素总数保持不变。本节使用 `mx.reshape(array, new_shape)`，不先交换B/T轴，也不排序或打乱其中任何一侧。

元素数量必须对得上：

$$
B\times T\times V=N\times V,\qquad B\times T=N.
$$

- logits的目标shape为 `(N,V)`：N个监督位置，每个位置保留V个类别分数。
- targets的目标shape为 `(N,)`：一维数组，含N个整数ID。`(N,)` 中的逗号用于构成Python单元素tuple，不表示新增一个轴。
- logits与targets必须使用同一位置顺序。仅仅shape相容，并不能挽救“只重排了一侧”的语义错位。

这里描述逻辑元素与shape的关系，不承诺reshape总是零拷贝。把V也合并成一个长向量会丢掉本节所需的“每行V个候选”接口。

主代码写出明确的N。以后若看到 `(-1,V)` 或 `(-1,)`，其中的 `-1` 表示按总元素数推断一个维度，最多只能有一个；本例推断结果就是N。这个shape中的 `-1` 与交叉熵 `axis=-1` 表示“最后一轴”的用途不同。

[MLX reshape API](https://ml-explore.github.io/mlx/build/html/python/_autosummary/mlx.core.reshape.html) · [固定v0.32.2 shape推断源码](https://github.com/ml-explore/mlx/blob/v0.32.2/mlx/primitives.cpp)

In [ ]:
# USER-RUN RESHAPE — prepared only, NOT executed in the master.

flat_logits = mx.reshape(toy_logits, (N, V))  # (4, 3)
flat_targets = mx.reshape(Y, (N,))            # (4,)

print('flat logits shape:', flat_logits.shape)
print('flat targets shape:', flat_targets.shape)
print('flat target IDs:', flat_targets)

# Hand-derived target order: [2, 1, 0, 2], with r = b*T + t.
# The master contains no runtime output.


## 4. 同一组逐位置交叉熵，换成N行的表示

17.1中，`nn.losses.cross_entropy(logits, Y, axis=-1, reduction='none')` 已能直接接受 `(B,T,V)` 与 `(B,T)`，返回 `(B,T)`。**MLX没有要求必须先展平；本节把监督位置显式列成N行，是为了看清每个target怎样参与loss。**

现在沿用同一个接口：

| 参数 | 本节输入 / 作用 |
|---|---|
| logits | `flat_logits:(N,V)`，保持raw scores |
| targets | `flat_targets:(N,)`，保持整数类别IDs |
| `axis=-1` | 类别轴仍是最后的V轴 |
| `reduction='none'` | 保留每个监督位置的一个loss |
| 返回值 | `flat_losses:(N,)` |

类别轴上的交叉熵计算与17.1相同：对每一行，计算正确类别的负对数概率。由于每一行的分数向量和正确ID都未变，数学上：

$$
\ell^{\text{flat}}_{bT+t}=\ell_{b,t}.
$$

因此展平不会改变某个位置对应的损失定义。这里继续直接传raw logits，不先softmax，不取argmax，也不再次shift Y。

下方另外用同一个reshape API把 `flat_losses` 放回 `(B,T)`，命名为 `loss_grid`，便于定位原来的行和时间位置。这是在重排同一组结果，不是又计算一种新loss。

[固定MLX v0.32.2 loss源码](https://github.com/ml-explore/mlx/blob/v0.32.2/python/mlx/nn/losses.py) · [cross_entropy API](https://ml-explore.github.io/mlx/build/html/python/nn/_autosummary_functions/mlx.nn.losses.cross_entropy.html)

In [ ]:
# USER-RUN TOKEN LOSSES — prepared only, NOT executed in the master.

flat_losses = nn.losses.cross_entropy(
    flat_logits,
    flat_targets,
    axis=-1,
    reduction='none',
)  # (N,) = (4,)

loss_grid = mx.reshape(flat_losses, (B, T))  # (2, 2)

print('flat loss shape:', flat_losses.shape)
print('loss grid shape:', loss_grid.shape)
print('flat losses:', flat_losses)

# Mathematical correspondence only: flat_losses[b*T+t] pairs with Y[b,t].
# No comparison test or lesson computation has been executed.


### 手算：展平前后，哪一个loss属于哪一个位置？

沿用17.1，raw logits为 \(\ln[a,b,c]\) 时，softmax分子为a、b、c；正确类别概率记为 \(p_y\)，位置loss为 \(-\ln p_y\)。

| r / 原位置 | 正确y | 正确类别概率 \(p_y\) | 该位置loss |
|---|---:|---|---|
| 0 / (0,0) | 2 | \(3/(1+2+3)=1/2\) | \(\ln2\) |
| 1 / (0,1) | 1 | \(2/(3+2+1)=1/3\) | \(\ln3\) |
| 2 / (1,0) | 0 | \(1/(1+1+2)=1/4\) | \(\ln4\) |
| 3 / (1,1) | 2 | \(1/(3+2+1)=1/6\) | \(\ln6\) |

所以同一组结果的两种组织方式在数学上为：

$$
\text{flat\_losses}=[\ln2,\ln3,\ln4,\ln6],
\qquad \text{shape }(4,),
$$

$$
\text{loss\_grid}=
\begin{bmatrix}
\ln2 & \ln3\\
\ln4 & \ln6
\end{bmatrix},
\qquad \text{shape }(2,2).
$$

这是由明确输入与公式得到的手算预期，未执行Notebook、数值实验或比较测试。不同浮点计算形式不在这里承诺逐bit一致。

## 5. 一个Batch中的多个目标，怎样汇总为一个标量？

本例有N=4个有效监督位置，每个位置等权。**Arithmetic mean（算术平均）** 的含义是：把这些位置的loss相加，再除以位置数N。

用 \(\ell_r\) 表示展平后第r个位置的loss；用L表示整批的平均loss。\(\sum_{r=0}^{N-1}\) 表示从第0个位置加到第N−1个位置：

$$
L=\frac1N\sum_{r=0}^{N-1}\ell_r.
$$

也可以保留B、T的下标：先把同一行的T个位置相加，再把B行相加，这就是下面的双重求和：

$$
L=\frac1{BT}\sum_{b=0}^{B-1}\sum_{t=0}^{T-1}\ell_{b,t}.
$$

两个式子遍历同一批位置，分母都是N=B×T。本例：

$$
L=\frac{\ln2+\ln3+\ln4+\ln6}{4}
\approx1.2425.
$$

这里平均的是已经算好的N个**位置损失**；V个候选类别已经参与各位置的交叉熵计算，平均时不再除以V。也不先平均logits，因为每组分数有自己对应的target。

### 代码前：`mx.mean` 与 Scalar

`mx.mean(array)` 对数组全部元素求算术平均；默认 `axis=None`，`keepdims=False`。当前 `flat_losses` 是长度N的一维数组，所以它返回一个 **scalar（标量）**，shape为 `()`。

`()` 表示没有轴的单个数；`(1,)` 则是长度为1的一维数组，两者shape不同。返回的 `mean_loss` 仍是MLX数组，这里不转换成Python浮点数。

[MLX mean API](https://ml-explore.github.io/mlx/build/html/python/_autosummary/mlx.core.mean.html)

In [ ]:
# USER-RUN TOKEN MEAN — prepared only, NOT executed in the master.

mean_loss = mx.mean(flat_losses)  # scalar MLX array, shape ()

print('mean loss shape:', mean_loss.shape)
print('mean loss:', mean_loss)

# Hand-derived expectation: (ln(2)+ln(3)+ln(4)+ln(6))/4, about 1.2425.
# This is prepared loss organization only, not a training step or runtime result.


### “一个训练步骤含多个Token-Level Targets”在本节意味着什么？

后续训练步骤会使用一个batch的监督信号。本节只说明供其使用的loss如何组织：一个B行、每行T个位置的batch，提供N=B×T个正确类别；这些位置的损失可共同组成一个平均标量L。

这个例子中，有2行序列、4个监督位置，虽然target ID 2出现了两次，仍有4项loss。4个监督项不等于要执行4次参数更新；把它们写成一个标量，也不会把它们变成只有一个位置的监督。

三个操作各负责一件事：

| 操作 | Shape变化 | 作用 |
|---|---|---|
| reshape logits / targets | `(B,T,V)→(N,V)`；`(B,T)→(N,)` | 按相同顺序组织N个监督位置 |
| cross-entropy，`reduction='none'` | `(N,V)` + `(N,)` → `(N,)` | 每个位置产生一个loss |
| mean | `(N,)→()` | 将这些位置loss等权汇总为一个标量 |

模型输入仍按原序列组织；合并的是loss端的监督位置轴。本节到这个标量接口为止，不讲参数树、梯度、`nn.value_and_grad`、optimizer或training loop。

## Work Copy 的可操作顺序与本节终点

之后需要使用时：

1. 在仓库根目录更新；用时间戳新文件名与 `cp -n` 将73母版复制到已有的 `notebooks/work/`，只打开新work副本。
2. 选择项目Python3.12 / `.venv` kernel，从本节imports开始。
3. 阅读监督单位与toy例子，再使用自带toy setup；X、Y、logits、B/T/V/N都在这一节定义。
4. 先读行号映射和reshape说明，再使用两侧的reshape代码。
5. 读逐位置交叉熵对应关系，再使用 `flat_losses` / `loss_grid` 代码；最后读等权mean并使用标量loss代码。

以上是未来work副本的操作说明。本轮没有读取或下载语料，没有执行Notebook、测试、训练、实验、采样或用户工作副本，也没有代答练习。手算预期、静态审阅与材料发布都不表示学员掌握或运行通过。

**本轮仅准备17.2，停在本节，不进入17.3。** 已通过课节保持原样。

累计实际修正：16.1=**1/2**、16.2=**1/2**、16.3至16.5及17.1各**0/2**、17.2初始生成=**0/2**。每节最多两次实际修正，跨轮与换对话不重置；后续只按请求逐小节至大Step20全部完成后停止。

## 参考与课程衔接

- [17.1 Logits、概率与交叉熵母版](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/1bfe7f8185664003941686510ef86d86322c3c02/notebooks/72_step17_1_logits_probabilities_cross_entropy.ipynb)：raw logits、整数targets与逐位置loss。
- [16.3 Batch sampler](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/1bfe7f8185664003941686510ef86d86322c3c02/notebooks/69_step16_3_batch_sampler.ipynb)：已经右移的 `X/Y:(B,T)`。
- [MLX reshape](https://ml-explore.github.io/mlx/build/html/python/_autosummary/mlx.core.reshape.html)、[mean](https://ml-explore.github.io/mlx/build/html/python/_autosummary/mlx.core.mean.html)、[cross_entropy](https://ml-explore.github.io/mlx/build/html/python/nn/_autosummary_functions/mlx.nn.losses.cross_entropy.html)：API含义。
- [MLX v0.32.2 ops.cpp](https://github.com/ml-explore/mlx/blob/v0.32.2/python/src/ops.cpp)、[primitives.cpp](https://github.com/ml-explore/mlx/blob/v0.32.2/mlx/primitives.cpp)、[losses.py](https://github.com/ml-explore/mlx/blob/v0.32.2/python/mlx/nn/losses.py)：固定版本接口、shape推断与交叉熵依据。在线文档可能更新，项目仍固定MLX0.32.2。
- [Step17课程文档](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/main/docs/step-17-language-model-loss.md) 与 [course roadmap](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/main/docs/course-roadmap.md)。